# AdaptiveDeltaXAI Master v1.9 — DRAFT

Not release-validated. Do not use Run all. Execute only the setup and tests until code review is complete.


In [1]:
from google.colab import drive
from pathlib import Path
import base64, zipfile
drive.mount('/content/drive')
DATA_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr')
RESULTS_ROOT = DATA_ROOT / 'results_master_v1_9'
REPO_ROOT = Path('/content/AdaptiveDeltaXAI_v1_9')
SOURCE_ZIP = DATA_ROOT / 'AdaptiveDeltaXAI_v1_9_source.zip'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
assert DATA_ROOT.exists(), f'Missing data folder: {DATA_ROOT}'
if not SOURCE_ZIP.exists(): SOURCE_ZIP.write_bytes(base64.b64decode("""..."""))
with zipfile.ZipFile(SOURCE_ZIP) as z: z.extractall(REPO_ROOT)
print('[PASS] setup complete:', RESULTS_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[PASS] setup complete: /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9


In [4]:
from pathlib import Path
import os, subprocess, sys
DATA_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr')
RESULTS_ROOT = DATA_ROOT / 'results_master_v1_9'
REPO_ROOT = Path('/content/AdaptiveDeltaXAI_v1_9')
assert REPO_ROOT.exists(), 'Run setup first.'
def run_stage(stage, *extra, allow_cpu_smoke=False):
    cmd=[sys.executable,str(REPO_ROOT/'scripts'/'run_v1_9_stage.py'),'--data-root',str(DATA_ROOT),'--output-root',str(RESULTS_ROOT),'--stage',stage,*map(str,extra)]
    if allow_cpu_smoke: cmd.append('--allow-cpu-smoke')
    env=dict(os.environ); env['PYTHONPATH']=str(REPO_ROOT/'src')+':'+str(REPO_ROOT/'scripts')
    print('RUN:', ' '.join(cmd), flush=True)
    p=subprocess.run(cmd,cwd=REPO_ROOT,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    print(p.stdout,end='',flush=True)
    if p.returncode: raise RuntimeError(f'Stage {stage} failed; diagnostic printed above.')
print('[PASS] self-contained runner ready; every stage exits and releases RAM.')


[PASS] self-contained runner ready; every stage exits and releases RAM.


In [5]:
from pathlib import Path

init_path = REPO_ROOT / "src" / "adaptive_deltaxai" / "__init__.py"

# Show the current package initialization before changing it.
original = init_path.read_text()
print("Current contents:\n", original)

# Remove only the import of the missing controller module.
lines = original.splitlines(keepends=True)
updated = "".join(
    line for line in lines
    if not line.strip().startswith("from .controller import ")
)

if updated == original:
    raise RuntimeError(
        "The expected controller import was not found. "
        "Stop and share the output above."
    )

init_path.write_text(updated)
print("\n[PASS] Missing controller import removed.")

Current contents:
 """Canonical v1.6 AdaptiveDeltaXAI implementation."""

from .swing import multipath_integrated_gradients, swing_attribution, linear_completeness_check

__all__ = ["multipath_integrated_gradients", "swing_attribution", "linear_completeness_check",
           "AdaptiveSwingController", "StepRecord", "summarize"]



RuntimeError: The expected controller import was not found. Stop and share the output above.

In [4]:
import torch
print('CUDA available:',torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
run_stage('preflight',allow_cpu_smoke=True)


CUDA available: True
GPU: Tesla T4
RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage preflight --allow-cpu-smoke
[START] stage=preflight device=cuda seed=42
[PASS] preflight receipt written: /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9/PRECHECK_v1_9.json
[DONE] stage=preflight elapsed_min=0.02


In [7]:
# 4. Unit tests: exact target/index, exact 26-history, cache age.
import subprocess,os,sys
env=dict(os.environ); env['PYTHONPATH']=str(REPO_ROOT/'src')+':'+str(REPO_ROOT/'scripts')
p=subprocess.run([sys.executable,'-m','pytest','-q','tests/test_swing.py','tests/test_swing_independent_reference.py','tests/test_v1_9_temporal_contract.py'],cwd=REPO_ROOT,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(p.stdout)
assert p.returncode==0


........                                                                 [100%]
8 passed in 3.67s



In [9]:
# prerun before #5
from pathlib import Path

core_path = REPO_ROOT / "scripts" / "core_v1_9.py"
source = core_path.read_text()

old = (
    "w=cfg['window']; lag=cfg['swing_lag']; "
    "hist=stream[r.t-(w+1+lag)+1:r.t+1]; "
    "fresh,_=fresh_swing(model,hist,cfg,device)"
)

new = (
    "w=cfg['window']; lag=cfg['swing_lag']; "
    "history_length=2*w+1+lag; "
    "hist=stream[r.t-history_length+1:r.t+1]; "
    "fresh,_=fresh_swing(model,hist,cfg,device)"
)

if old not in source:
    raise RuntimeError(
        "Expected code was not found exactly. "
        "Do not rerun Cell 5; send me this output."
    )

core_path.write_text(source.replace(old, new, 1))
print("[PASS] Cache-fidelity fresh map now uses the controller's 46-observation history.")

[PASS] Cache-fidelity fresh map now uses the controller's 46-observation history.


In [10]:
# 5. Train/full monitoring: 6 memory-bounded jobs.
for seed in (42,52,62):
  for stream in ('temperature','power'): run_stage('synthetic','--seed',seed,'--stream',stream)


RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage synthetic --seed 42 --stream temperature
[START] stage=synthetic device=cuda seed=42
[RUN] synthetic stream=temperature, seed=42
  epoch 1/5 val_loss=0.60295
  epoch 2/5 val_loss=0.34780
  epoch 3/5 val_loss=0.28643
  epoch 4/5 val_loss=0.31881
  epoch 5/5 val_loss=0.28093
[DONE] stage=synthetic elapsed_min=1.75
RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage synthetic --seed 42 --stream power
[START] stage=synthetic device=cuda seed=42
[RUN] synthetic stream=power, seed=42
  epoch 1/5 val_loss=0.38919
  epoch 2/5 val_loss=0.19176
  epoch 3/5 v

In [5]:
# pre-cell
from pathlib import Path

results = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "ADXAI_Dataset_48hr/results_master_v1_9"
)

for seed in (42, 52, 62):
    for stream in ("temperature", "power"):
        file = results / f"synthetic_{stream}_seed{seed}.json"
        print("SAVED" if file.exists() else "MISSING", file.name)

SAVED synthetic_temperature_seed42.json
SAVED synthetic_power_seed42.json
SAVED synthetic_temperature_seed52.json
SAVED synthetic_power_seed52.json
SAVED synthetic_temperature_seed62.json
SAVED synthetic_power_seed62.json


In [7]:
# fixes
# Restore the two temporary v1.9 fixes after reconnecting.

# Fix 1: remove the import of the missing controller module.
init_path = REPO_ROOT / "src" / "adaptive_deltaxai" / "__init__.py"
original = init_path.read_text()
updated = "".join(
    line for line in original.splitlines(keepends=True)
    if not line.strip().startswith("from .controller import ")
)

if updated != original:
    init_path.write_text(updated)
    print("[PASS] Missing controller import removed.")
else:
    print("[CHECK] Controller import is already absent.")

# Fix 2: use the controller's 46-observation history in cache fidelity.
core_path = REPO_ROOT / "scripts" / "core_v1_9.py"
source = core_path.read_text()

old = (
    "w=cfg['window']; lag=cfg['swing_lag']; "
    "hist=stream[r.t-(w+1+lag)+1:r.t+1]; "
    "fresh,_=fresh_swing(model,hist,cfg,device)"
)

new = (
    "w=cfg['window']; lag=cfg['swing_lag']; "
    "history_length=2*w+1+lag; "
    "hist=stream[r.t-history_length+1:r.t+1]; "
    "fresh,_=fresh_swing(model,hist,cfg,device)"
)

if old in source:
    core_path.write_text(source.replace(old, new, 1))
    print("[PASS] Cache-fidelity history fix applied.")
elif new in source:
    print("[PASS] Cache-fidelity history fix is already present.")
else:
    raise RuntimeError(
        "Could not verify the cache-fidelity fix. Stop and share this output."
    )

[CHECK] Controller import is already absent.
[PASS] Cache-fidelity history fix applied.


In [8]:
# 6. Detection, imbalance, event, and refresh coverage.
for seed in (42,52,62):
  for stream in ('temperature','power'): run_stage('synthetic_metrics','--seed',seed,'--stream',stream)


RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage synthetic_metrics --seed 42 --stream temperature
[START] stage=synthetic_metrics device=cuda seed=42
[SKIP] synthetic_temperature_seed42_extra_metrics.csv
[DONE] stage=synthetic_metrics elapsed_min=0.00
RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage synthetic_metrics --seed 42 --stream power
[START] stage=synthetic_metrics device=cuda seed=42
[SKIP] synthetic_power_seed42_extra_metrics.csv
[DONE] stage=synthetic_metrics elapsed_min=0.00
RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/My

In [10]:
# pre-cell hot fix
import sys
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "lime"],
    check=True
)

print("[PASS] LIME installation completed.")



[PASS] LIME installation completed.


In [11]:
# 7. Supplementary current-score proxy faithfulness.
run_stage('faithfulness')


RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage faithfulness
[START] stage=faithfulness device=cuda seed=42

100%|██████████| 1/1 [00:00<00:00, 16.98it/s]

100%|██████████| 1/1 [00:00<00:00, 32.71it/s]

100%|██████████| 1/1 [00:00<00:00, 32.82it/s]

100%|██████████| 1/1 [00:00<00:00, 33.07it/s]

100%|██████████| 1/1 [00:00<00:00, 32.63it/s]

100%|██████████| 1/1 [00:00<00:00, 33.15it/s]

100%|██████████| 1/1 [00:00<00:00, 31.88it/s]

100%|██████████| 1/1 [00:00<00:00, 32.84it/s]

100%|██████████| 1/1 [00:00<00:00, 29.77it/s]

100%|██████████| 1/1 [00:00<00:00, 30.82it/s]
  faithfulness 10/50

100%|██████████| 1/1 [00:00<00:00, 31.32it/s]

100%|██████████| 1/1 [00:00<00:00, 31.12it/s]

100%|██████████| 1/1 [00:00<00:00, 31.07it/s]

100%|██████████| 1/1 [00:00<00:00, 28.18it/s]

100%|██████

In [12]:
# 8. Primary direct paired-window prediction-difference faithfulness.
run_stage('delta_faithfulness')


RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage delta_faithfulness
[START] stage=delta_faithfulness device=cuda seed=42

100%|██████████| 1/1 [00:00<00:00, 21.02it/s]

100%|██████████| 1/1 [00:00<00:00, 23.97it/s]

100%|██████████| 1/1 [00:00<00:00, 24.28it/s]

100%|██████████| 1/1 [00:00<00:00, 24.86it/s]

100%|██████████| 1/1 [00:00<00:00, 18.57it/s]

100%|██████████| 1/1 [00:00<00:00, 22.52it/s]

100%|██████████| 1/1 [00:00<00:00, 24.44it/s]

100%|██████████| 1/1 [00:00<00:00, 24.45it/s]

100%|██████████| 1/1 [00:00<00:00, 23.74it/s]

100%|██████████| 1/1 [00:00<00:00, 24.09it/s]
  direct delta faithfulness 10/50

100%|██████████| 1/1 [00:00<00:00, 25.92it/s]

100%|██████████| 1/1 [00:00<00:00, 25.19it/s]

100%|██████████| 1/1 [00:00<00:00, 24.40it/s]

100%|██████████| 1/1 [00:00<00:00

In [13]:
# 9. Three-seed exact-matched adaptive/periodic/random/offline-oracle policies.
for seed in (42,52,62):
  for policy in ('adaptive','periodic','random','score_change'): run_stage('policy','--seed',seed,'--policy',policy)


RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage policy --seed 42 --policy adaptive
[START] stage=policy device=cuda seed=42
[DONE] stage=policy elapsed_min=0.01
RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage policy --seed 42 --policy periodic
[START] stage=policy device=cuda seed=42
[DONE] stage=policy elapsed_min=1.70
RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage policy --seed 42 --policy r

In [14]:
# 10. Three-seed six-component ablation. No-I1 is the always-refresh reference.
for seed in (42,52,62):
  for v in ('Full','No_I1','No_I2','No_I3','No_I4','No_I5'): run_stage('ablation','--seed',seed,'--variant',v)


RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage ablation --seed 42 --variant Full
[START] stage=ablation device=cuda seed=42
[DONE] stage=ablation elapsed_min=0.36
RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage ablation --seed 42 --variant No_I1
[START] stage=ablation device=cuda seed=42
[DONE] stage=ablation elapsed_min=3.83
RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage ablation --seed 42 -

In [15]:
# 11. Repeated warmed T4 batching x caching timing.
for c in ('unbatched_always_refresh','unbatched_adaptive_refresh','batched_always_refresh','batched_adaptive_refresh'): run_stage('timing','--condition',c)


RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage timing --condition unbatched_always_refresh
[START] stage=timing device=cuda seed=42
[DONE] stage=timing elapsed_min=26.73
RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage timing --condition unbatched_adaptive_refresh
[START] stage=timing device=cuda seed=42
[DONE] stage=timing elapsed_min=5.91
RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage timing

In [16]:
# 12. Comparable T4 method latency.
for m in ('zero_ig','aw_ig','full_swing','kernel_shap','lime'): run_stage('method_latency','--method',m)


RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage method_latency --method zero_ig
[START] stage=method_latency device=cuda seed=42
[DONE] stage=method_latency elapsed_min=0.02
RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage method_latency --method aw_ig
[START] stage=method_latency device=cuda seed=42
[DONE] stage=method_latency elapsed_min=0.02
RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage met

In [17]:
# 13. Three-seed 8/16/32/64 scaling.
for seed in (42,52,62):
  for d in (8,16,32,64): run_stage('scaling','--seed',seed,'--dimensions',d)


RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage scaling --seed 42 --dimensions 8
[START] stage=scaling device=cuda seed=42
  epoch 1/5 val_loss=0.60290
  epoch 2/5 val_loss=0.34772
  epoch 3/5 val_loss=0.28643
  epoch 4/5 val_loss=0.31884
  epoch 5/5 val_loss=0.28090
[DONE] stage=scaling elapsed_min=0.49
RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage scaling --seed 42 --dimensions 16
[START] stage=scaling device=cuda seed=42
  epoch 1/5 val_loss=0.70505
  epoch 2/5 val_loss=0.32594
  epoch 3/5 val_loss=0.26654
  epoch 4/5 val_loss=0.30235
  epoch 5/5 val_loss=0.25580
[DONE] stage=scaling el

In [18]:
# 14. Three-seed GRU/Transformer validation.
for seed in (42,52,62): run_stage('architecture','--seed',seed)


RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage architecture --seed 42
[START] stage=architecture device=cuda seed=42
  epoch 1/5 val_loss=0.46406
  epoch 2/5 val_loss=0.31543
  epoch 3/5 val_loss=0.27642
  epoch 4/5 val_loss=0.30189
  epoch 5/5 val_loss=0.25578
[DONE] stage=architecture elapsed_min=3.09
RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage architecture --seed 52
[START] stage=architecture device=cuda seed=52
  epoch 1/5 val_loss=0.33654
  epoch 2/5 val_loss=0.47316
  epoch 3/5 val_loss=0.30219
  epoch 4/5 val_loss=0.27043
  epoch 5/5 val_loss=0.27009
[DONE] stage=architecture ela

In [19]:
# 15. Corrected public subset; one unit per process.
for u in ('machine-1-1','machine-1-2','machine-1-3'): run_stage('public','--dataset','SMD','--unit',u,'--seed',42)
run_stage('public','--dataset','PSM','--unit','PSM','--seed',42)


RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage public --dataset SMD --unit machine-1-1 --seed 42
[START] stage=public device=cuda seed=42
  reconstruction epoch 1/5
  reconstruction epoch 2/5
  reconstruction epoch 3/5
  reconstruction epoch 4/5
  reconstruction epoch 5/5
  SMD/machine-1-1/seed42: 10/50
  SMD/machine-1-1/seed42: 20/50
  SMD/machine-1-1/seed42: 30/50
  SMD/machine-1-1/seed42: 40/50
  SMD/machine-1-1/seed42: 50/50
[DONE] stage=public elapsed_min=0.22
RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage public --dataset SMD --unit machine-1-2 --seed 42
[START] stage=public device=c

In [20]:
# 16. Supplementary public calibration, never tuned on test labels.
for u in ('machine-1-1','machine-1-2','machine-1-3'): run_stage('public_calibrated','--dataset','SMD','--unit',u,'--seed',42)
run_stage('public_calibrated','--dataset','PSM','--unit','PSM','--seed',42)


RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage public_calibrated --dataset SMD --unit machine-1-1 --seed 42
[START] stage=public_calibrated device=cuda seed=42
  reconstruction epoch 1/5
  reconstruction epoch 2/5
  reconstruction epoch 3/5
  reconstruction epoch 4/5
  reconstruction epoch 5/5
[DONE] stage=public_calibrated elapsed_min=0.07
RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage public_calibrated --dataset SMD --unit machine-1-2 --seed 42
[START] stage=public_calibrated device=cuda seed=42
  reconstruction epoch 1/5
  reconstruction epoch 2/5
  reconstruction epoch 3/5
  reconstruc

In [22]:
# hot fix precell 17
import pandas as pd

methods = ("zero_ig", "aw_ig", "full_swing", "kernel_shap", "lime")

files = [
    RESULTS_ROOT / f"method_latency_{method}_seed42.csv"
    for method in methods
]

missing = [path.name for path in files if not path.is_file()]

if missing:
    print("Missing files:")
    for name in missing:
        print("  ", name)
    raise FileNotFoundError(
        "One or more Cell 12 results are missing. "
        "Stop and send this output before rerunning anything."
    )

frames = []
for method, path in zip(methods, files):
    frame = pd.read_csv(path)

    if frame.empty or "method" not in frame.columns:
        raise ValueError(f"{path.name} is empty or missing the method column.")

    if set(frame["method"]) != {method}:
        raise ValueError(f"{path.name} contains an unexpected method.")

    print(f"[FOUND] {path.name}: {len(frame)} rows")
    frames.append(frame)

combined_path = RESULTS_ROOT / "method_latency_trials.csv"

if combined_path.exists():
    raise FileExistsError(
        f"{combined_path.name} already exists. Stop and share this output "
        "so we can inspect it without overwriting it."
    )

combined = pd.concat(frames, ignore_index=True)
combined.to_csv(combined_path, index=False)

print(f"[PASS] Created {combined_path.name}: {len(combined)} rows")

[FOUND] method_latency_zero_ig_seed42.csv: 50 rows
[FOUND] method_latency_aw_ig_seed42.csv: 50 rows
[FOUND] method_latency_full_swing_seed42.csv: 50 rows
[FOUND] method_latency_kernel_shap_seed42.csv: 50 rows
[FOUND] method_latency_lime_seed42.csv: 50 rows
[PASS] Created method_latency_trials.csv: 250 rows


In [23]:
# 17. Post-hoc paired intervals and latency tails.
run_stage('posthoc')


RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage posthoc
[START] stage=posthoc device=cuda seed=42
[PASS] posthoc metrics saved; event_clustered_bootstrap=True
[DONE] stage=posthoc elapsed_min=0.00


In [2]:
# hot fix pre 18
from pathlib import Path

results = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "ADXAI_Dataset_48hr/results_master_v1_9"
)

print("Results folder exists:", results.exists())

if results.exists():
    for path in sorted(results.iterdir()):
        if path.is_file():
            size_mb = path.stat().st_size / (1024**2)
            print(f"{size_mb:8.2f} MB  {path.name}")

Results folder exists: True
    0.00 MB  CONFIG_v1_9.json
    0.00 MB  FEATURE_MANIFEST_v1_9.json
    0.00 MB  HARDWARE_PROFILE_v1_9.json
    0.03 MB  PAPER_TABLES_v1_9.md
    0.00 MB  PRECHECK_v1_9.json
    0.00 MB  ablation_Full_seed42.json
    0.00 MB  ablation_Full_seed52.json
    0.00 MB  ablation_Full_seed62.json
    0.00 MB  ablation_No_I1_seed42.json
    0.00 MB  ablation_No_I1_seed52.json
    0.00 MB  ablation_No_I1_seed62.json
    0.00 MB  ablation_No_I2_seed42.json
    0.00 MB  ablation_No_I2_seed52.json
    0.00 MB  ablation_No_I2_seed62.json
    0.00 MB  ablation_No_I3_seed42.json
    0.00 MB  ablation_No_I3_seed52.json
    0.00 MB  ablation_No_I3_seed62.json
    0.00 MB  ablation_No_I4_seed42.json
    0.00 MB  ablation_No_I4_seed52.json
    0.00 MB  ablation_No_I4_seed62.json
    0.00 MB  ablation_No_I5_seed42.json
    0.00 MB  ablation_No_I5_seed52.json
    0.00 MB  ablation_No_I5_seed62.json
    0.00 MB  ablation_per_seed.csv
    0.00 MB  ablation_summary.csv
    0.00 M

In [3]:
# hot fix #2 before 18
from pathlib import Path

# The setup cell may have restored the missing controller import.
init_path = REPO_ROOT / "src" / "adaptive_deltaxai" / "__init__.py"
original_init = init_path.read_text()

updated_init = "".join(
    line for line in original_init.splitlines(keepends=True)
    if not line.strip().startswith("from .controller import ")
)

if updated_init != original_init:
    init_path.write_text(updated_init)
    print("[PASS] Missing-controller import fix restored.")
else:
    print("[PASS] Missing-controller import is already absent.")

# Repair the controller-record loader used by Cell 18.
core_path = REPO_ROOT / "scripts" / "core_v1_9.py"
source = core_path.read_text()

old = (
    "def load_controller_records(path):\n"
    "    data=np.load(path, allow_pickle=False)\n"
    "    records=[]"
)

new = (
    "def load_controller_records(path):\n"
    "    # Read each compressed array once, rather than repeatedly per record.\n"
    "    with np.load(path, allow_pickle=False) as archive:\n"
    "        data = {key: archive[key] for key in archive.files}\n"
    "    records=[]"
)

if old in source:
    core_path.write_text(source.replace(old, new, 1))
    print("[PASS] Memory-efficient controller loader applied.")
elif new in source:
    print("[PASS] Memory-efficient controller loader is already present.")
else:
    raise RuntimeError(
        "Could not find the expected loader code. "
        "Stop and send this output; do not rerun Cell 18."
    )

[PASS] Missing-controller import fix restored.
[PASS] Memory-efficient controller loader applied.


In [7]:
#hotfix #3
core_path = REPO_ROOT / "scripts" / "core_v1_9.py"
source = core_path.read_text()

start_marker = "def load_controller_records(path):"
end_marker = "\ndef load_controller_refresh_arrays(path):"

if start_marker not in source or end_marker not in source:
    raise RuntimeError(
        "Could not find the expected function boundaries. "
        "Stop and send me this output."
    )

before, rest = source.split(start_marker, 1)
function_body, after = rest.split(end_marker, 1)

# This change is limited to load_controller_records.
if "data.files" in function_body:
    function_body = function_body.replace("data.files", "data")
    repaired = before + start_marker + function_body + end_marker + after
    core_path.write_text(repaired)
    print("[PASS] Dictionary membership checks corrected.")
else:
    print("[CHECK] No data.files references found in this function.")

# Check that the updated file is valid Python syntax.
compile(core_path.read_text(), str(core_path), "exec")
print("[PASS] Core script syntax check passed.")

[PASS] Dictionary membership checks corrected.
[PASS] Core script syntax check passed.


In [8]:
# 18. Assemble evidence/tables/figures/checks.
run_stage('assemble')
print((RESULTS_ROOT/'CONSISTENCY_REPORT_v1_9.json').read_text())


RUN: /usr/bin/python3 /content/AdaptiveDeltaXAI_v1_9/scripts/run_v1_9_stage.py --data-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr --output-root /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9 --stage assemble
[START] stage=assemble device=cuda seed=42
[DONE] stage=assemble elapsed_min=0.05
{
  "status": "PASS",
  "three_seed_synthetic": true,
  "three_seed_synthetic_extra_metrics": true,
  "three_seed_refresh_coverage": true,
  "three_seed_ablation": true,
  "three_seed_scaling": true,
  "three_seed_architecture": true,
  "public_alignment_gate": true,
  "public_exact_budget_match": true,
  "public_completeness_gate": true,
  "public_threshold_free_metrics": true,
  "supplementary_public_calibration": true,
  "paired_aopc": true,
  "direct_prediction_difference_faithfulness": true,
  "latency_tails": true,
  "hardware": {
    "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
    "platform": "Linux-6.6.159+-x86_64-with-glibc

In [9]:
# 19. Freeze only after Cell 18 says status PASS.
import subprocess,os,sys
bundle=DATA_ROOT/'results_master_v1_9_frozen.zip'
env=dict(os.environ); env['PYTHONPATH']=str(REPO_ROOT/'src')+':'+str(REPO_ROOT/'scripts')
p=subprocess.run([sys.executable,str(REPO_ROOT/'scripts'/'freeze_results_v1_9.py'),'--results',str(RESULTS_ROOT),'--output',str(bundle)],cwd=REPO_ROOT,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(p.stdout)
assert p.returncode==0
print('Frozen:',bundle)


{
  "release": "Master v1.9",
  "bundle": "/content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9_frozen.zip",
  "sha256": "119ffebeca5d49cb863e31679a124ee8a916c3476439685ebcee23d97574c23a",
  "seeds": [
    42,
    52,
    62
  ],
  "status": "frozen",
  "hardware": {
    "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
    "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
    "cpu_model": "Intel(R) Xeon(R) CPU @ 2.00GHz",
    "memory_kb": 13286924,
    "torch": "2.14.1+cu130",
    "cuda_available": true,
    "cuda_version": "13.0",
    "gpu": "Tesla T4",
    "timing_device_label": "GPU"
  },
  "public_label_alignment": "window_end_t_minus_1"
}
Verified 15 MiB parts manifest: /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9_frozen.zip.parts_manifest.json

Frozen: /content/drive/MyDrive/Colab Notebooks/ADXAI_Dataset_48hr/results_master_v1_9_frozen.zip
